# Hypothesis H2
## Surface activities such as surfing have a higher recorded incident frequency but a lower fatality percentage than submerged activities such as scuba diving.
#### Business question
#### Which activities are associated with the highest number of recorded incidents, and how severe are those incidents?
(How do surface activities like surfing compare with submerged activities like scuba diving in recorded incident frequency and fatality percentage?)

In [1]:
import pandas as pd

In [2]:
shark_df = pd.read_csv("C:/Users/yurii/Downloads/IH_Projects/shark_attack/shark_clean_final.csv")

In [3]:
shark_df

,Country,State,Location,Activity,Date,Year,Type,Month,Sex,Age,Injury_Category,Time,Species,Fatal Y/N
0,Australia,Western Australia,Sorrento Beach Perth,Swimming,18 september,2026,Unprovoked,September,M,63.0,Other,10:15,White Shark,Y
1,Canada,Quebec,Off The Coast Of Perce Le Bilbo Dive Site,Diving,16 september,2026,Unprovoked,September,M,NaN,Injury,10:30,White Shark,N
2,Bahamas,Bimini,Bimini Island,Swimming,14 september,2026,Unprovoked,September,F,37.0,Injury,17:30,Other/Unknown,N
3,Australia,Western Australia,Geraldton,Surfing,13 september,2026,Unprovoked,September,M,NaN,Severe injury,09:45,Other/Unknown,N
4,Usa,Hawaii,Honolulu,Surfing,7 september,2026,Unprovoked,September,M,24.0,Other,16:40,Tiger Shark,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3993,Australia,Queensland,Harvey Bay,NaN,12-jan-1976,1976,Unprovoked,January,F,NaN,Other,No time specified,Other/Unknown,N
3994,South Africa,Western Cape Province,Kalk Bay,Fishing for snoek & yellowtail,11-jan-1976,1976,Provoked,January,Unknown,NaN,Other,07:30,White Shark,N
3995,Usa,Florida,"Off Fort Pierce, St Lucie County",Spearfishing / scuba diving,08-jan-1976,1976,Unprovoked,January,M,25.0,Injury,No time specified,Other/Unknown,N
3996,New Zealand,North Island,"Te Kaha, East Coast",Spearfishing,02-jan-1976,1976,Unprovoked,January,M,NaN,Fatal,13:00,Bronze Whaler Shark,Y


Analysis Plan
1. Count recorded incidents by activity
   - Identify which activities appear most often in the dataset.
2. Select the main activities
   - Focus on the top 8–10 activities and exclude very rare categories.
3. Calculate incident share (%)
   - Measure what percentage of all recorded incidents each activity represents.
4. Calculate fatality rate (%)
   - Compare the percentage of fatal incidents for each activity.
5. Analyze injury severity
   - Compare Injury_Category across activities.
6. Calculate Severe/Fatal rate (%)
   - Measure the share of Severe injury + Fatal cases for each activity.
7. Create a final comparison table
   Include:
   - Total Incidents
   - Incident %
   - Fatal Incidents
   - Fatality %
   - Severe/Fatal %
8. Business conclusion
   Identify which activities have the highest incident frequency and which have the most severe outcomes.

In [29]:
shark_df["Activity"] = shark_df["Activity"].replace(
    ["Scuba diving", "Scuba Diving"],
    "Scuba diving"
)

1 — Count recorded incidents by activity

In [31]:
activity_counts = shark_df["Activity"].value_counts()

activity_counts.head(10)

Activity
Surfing          1098
Swimming          568
Spearfishing      265
Fishing           238
Snorkeling        133
Wading            129
Diving             89
Scuba diving       82
Body boarding      63
Standing           62
Name: count, dtype: int64

2 — Select the top 10 activities

In [6]:
top_activities = activity_counts.head(10).index

top_activities

Index(['Surfing', 'Swimming', 'Spearfishing', 'Fishing', 'Snorkeling',
       'Wading', 'Diving', 'Scuba diving', 'Body boarding', 'Standing'],
      dtype='str', name='Activity')

In [36]:
activity_df = shark_df[
    shark_df["Activity"].isin(top_activities)
].copy()

activity_df.head(5)

,Country,State,Location,Activity,Date,Year,Type,Month,Sex,Age,Injury_Category,Time,Species,Fatal Y/N
0,Australia,Western Australia,Sorrento Beach Perth,Swimming,18 september,2026,Unprovoked,September,M,63.0,Other,10:15,White Shark,Y
1,Canada,Quebec,Off The Coast Of Perce Le Bilbo Dive Site,Diving,16 september,2026,Unprovoked,September,M,NaN,Injury,10:30,White Shark,N
2,Bahamas,Bimini,Bimini Island,Swimming,14 september,2026,Unprovoked,September,F,37.0,Injury,17:30,Other/Unknown,N
3,Australia,Western Australia,Geraldton,Surfing,13 september,2026,Unprovoked,September,M,NaN,Severe injury,09:45,Other/Unknown,N
4,Usa,Hawaii,Honolulu,Surfing,7 september,2026,Unprovoked,September,M,24.0,Other,16:40,Tiger Shark,N


3 — Calculate incident share (%)

In [37]:
incident_percentage = (
    activity_counts.head(10) / len(shark_df) * 100
)

incident_percentage = incident_percentage.round(2)

In [38]:
activity_summary = pd.DataFrame({
    "Total_Incidents": activity_counts.head(10),
    "Incident_Percentage": incident_percentage
})

activity_summary

,Total_Incidents,Incident_Percentage
Activity,,
Surfing,1098,27.46
Swimming,568,14.21
Spearfishing,265,6.63
Fishing,238,5.95
Snorkeling,133,3.33
Wading,129,3.23
Diving,89,2.23
Scuba diving,82,2.05
Body boarding,63,1.58


4 — Calculate fatality percentage by activity

In [39]:
activity_df["Fatal Y/N"].value_counts(dropna=False)

Fatal Y/N
N          2299
Y           272
UNKNOWN     156
Name: count, dtype: int64

In [40]:
known_fatality = activity_df[
    activity_df["Fatal Y/N"].isin(["Y", "N"])
].copy()

In [41]:
known_incidents = known_fatality.groupby("Activity").size()

known_incidents

Activity
Body boarding      63
Diving             79
Fishing           226
Scuba diving       67
Snorkeling        128
Spearfishing      249
Standing           61
Surfing          1060
Swimming          515
Wading            123
dtype: int64

In [42]:
fatal_incidents = (
    known_fatality[known_fatality["Fatal Y/N"] == "Y"]
    .groupby("Activity")
    .size()
)

fatal_incidents

Activity
Body boarding    16
Diving           18
Fishing          13
Scuba diving     13
Snorkeling       20
Spearfishing     35
Standing          1
Surfing          61
Swimming         94
Wading            1
dtype: int64

In [43]:
fatality_percentage = (
    fatal_incidents / known_incidents * 100
).fillna(0).round(2)

fatality_percentage

Activity
Body boarding    25.40
Diving           22.78
Fishing           5.75
Scuba diving     19.40
Snorkeling       15.62
Spearfishing     14.06
Standing          1.64
Surfing           5.75
Swimming         18.25
Wading            0.81
dtype: float64

In [44]:
activity_summary["Fatal_Incidents"] = fatal_incidents
activity_summary["Fatal_Incidents"] = (
    activity_summary["Fatal_Incidents"]
    .fillna(0)
    .astype(int)
)

activity_summary["Fatality_Percentage"] = fatality_percentage.fillna(0)

activity_summary

,Total_Incidents,Incident_Percentage,Fatal_Incidents,Fatality_Percentage
Activity,,,,
Surfing,1098,27.46,61,5.75
Swimming,568,14.21,94,18.25
Spearfishing,265,6.63,35,14.06
Fishing,238,5.95,13,5.75
Snorkeling,133,3.33,20,15.62
Wading,129,3.23,1,0.81
Diving,89,2.23,18,22.78
Scuba diving,82,2.05,13,19.40
Body boarding,63,1.58,16,25.40


5 — Analyze injury severity by activity

In [45]:
activity_df["Injury_Category"].value_counts(dropna=False)

Injury_Category
Injury           1629
No injury         308
Fatal             255
Minor injury      224
Severe injury     159
Other             133
Unknown            19
Name: count, dtype: int64

In [46]:
injury_by_activity = (
    activity_df
    .groupby(["Activity", "Injury_Category"])
    .size()
    .unstack(fill_value=0)
)

injury_by_activity

Injury_Category,Fatal,Injury,Minor injury,No injury,Other,Severe injury,Unknown
Activity,,,,,,,
Body boarding,16,24,3,8,2,10,0
Diving,16,41,6,9,6,6,5
Fishing,12,126,13,55,17,15,0
Scuba diving,17,35,3,11,13,3,0
Snorkeling,18,76,8,5,6,19,1
Spearfishing,28,153,14,29,20,19,2
Standing,1,47,10,1,1,2,0
Surfing,58,681,91,184,37,41,6
Swimming,88,349,56,6,24,40,5


In [48]:
injury_percentage = (
    injury_by_activity
    .div(injury_by_activity.sum(axis=1), axis=0)
    * 100
).round(2)

injury_percentage

Injury_Category,Fatal,Injury,Minor injury,No injury,Other,Severe injury,Unknown
Activity,,,,,,,
Body boarding,25.40,38.10,4.76,12.70,3.17,15.87,0.00
Diving,17.98,46.07,6.74,10.11,6.74,6.74,5.62
Fishing,5.04,52.94,5.46,23.11,7.14,6.30,0.00
Scuba diving,20.73,42.68,3.66,13.41,15.85,3.66,0.00
Snorkeling,13.53,57.14,6.02,3.76,4.51,14.29,0.75
Spearfishing,10.57,57.74,5.28,10.94,7.55,7.17,0.75
Standing,1.61,75.81,16.13,1.61,1.61,3.23,0.00
Surfing,5.28,62.02,8.29,16.76,3.37,3.73,0.55
Swimming,15.49,61.44,9.86,1.06,4.23,7.04,0.88


6 — Calculate Severe/Fatal percentage

In [49]:
severe_fatal_percentage = (
    injury_percentage["Severe injury"] +
    injury_percentage["Fatal"]
)

severe_fatal_percentage

Activity
Body boarding    41.27
Diving           24.72
Fishing          11.34
Scuba diving     24.39
Snorkeling       27.82
Spearfishing     17.74
Standing          4.84
Surfing           9.01
Swimming         22.53
Wading            3.88
dtype: float64

In [50]:
activity_summary["Severe_Fatal_Percentage"] = severe_fatal_percentage

activity_summary

,Total_Incidents,Incident_Percentage,Fatal_Incidents,Fatality_Percentage,Severe_Fatal_Percentage
Activity,,,,,
Surfing,1098,27.46,61,5.75,9.01
Swimming,568,14.21,94,18.25,22.53
Spearfishing,265,6.63,35,14.06,17.74
Fishing,238,5.95,13,5.75,11.34
Snorkeling,133,3.33,20,15.62,27.82
Wading,129,3.23,1,0.81,3.88
Diving,89,2.23,18,22.78,24.72
Scuba diving,82,2.05,13,19.40,24.39
Body boarding,63,1.58,16,25.40,41.27


7 — Sort and interpret the results

In [51]:
activity_by_incidents = activity_summary.sort_values(
    by="Total_Incidents",
    ascending=False
)

activity_by_incidents

,Total_Incidents,Incident_Percentage,Fatal_Incidents,Fatality_Percentage,Severe_Fatal_Percentage
Activity,,,,,
Surfing,1098,27.46,61,5.75,9.01
Swimming,568,14.21,94,18.25,22.53
Spearfishing,265,6.63,35,14.06,17.74
Fishing,238,5.95,13,5.75,11.34
Snorkeling,133,3.33,20,15.62,27.82
Wading,129,3.23,1,0.81,3.88
Diving,89,2.23,18,22.78,24.72
Scuba diving,82,2.05,13,19.40,24.39
Body boarding,63,1.58,16,25.40,41.27


In [22]:
activity_by_severity = activity_summary.sort_values(
    by="Severe_Fatal_Percentage",
    ascending=False
)

activity_by_severity

,Total_Incidents,Incident_Percentage,Fatal_Incidents,Fatality_Percentage,Severe_Fatal_Percentage
Activity,,,,,
Body boarding,63,1.58,16,25.40,41.27
Snorkeling,133,3.33,20,15.62,27.82
Diving,89,2.23,18,22.78,24.72
Scuba diving,82,2.05,13,19.40,24.39
Swimming,568,14.21,94,18.25,22.53
Spearfishing,265,6.63,35,14.06,17.74
Fishing,238,5.95,13,5.75,11.34
Surfing,1098,27.46,61,5.75,9.01
Standing,62,1.55,1,1.64,4.84


- Surfing has by far the highest number of recorded incidents: 1,098 cases, or 27.46% of all incidents. However, its severe/fatal outcome rate is relatively low at 9.01%.

- Swimming combines high incident frequency with high severity: 568 incidents and a 22.53% severe/fatal rate.

- Body boarding has relatively few incidents (63), but the highest severity: 41.27% of cases resulted in severe or fatal outcomes.

- Snorkeling, diving and scuba diving also show relatively high severity, with severe/fatal rates around 24–28%, despite much lower incident counts.

Business conclusion:

- SharkSafe should not evaluate activities based only on incident frequency. 
- Surfing requires strong preventive guidance because incidents are frequent, while swimming, body boarding, snorkeling and diving require stricter safety protocols because their recorded incidents tend to be more severe.